# wh_poc — 장소 추천 에이전트

## 구조

```
Layer 0  공용 유틸        dump / _run_id / _load_json / _log / _strip_tags
Layer 1  외부 클라이언트   _kakao_keyword / _naver_search / scrape_urls / _llm / _llm_async
Layer 2  툴 (에이전트 노출) get_today / collect_places / collect_info / collect_context / read_file
Layer 3  에이전트          LlmAgent + Runner
```

## 설계 원칙

- **툴 경계 = 에이전트가 판단해야 하는 지점.** 코드가 관측 가능한 조건(결과 개수, `is_end`,
  API 응답 상태)은 함수 안으로 내리고, 의미론적 판단만 에이전트에게 남긴다.
- **모든 툴 결과는 파일로 떨구고 경로만 반환.** 원본이 컨텍스트에 올라가지 않으므로
  할루시네이션 여지가 줄고, 에이전트는 `read_file`로 필요할 때만 읽는다.
- **한 곳에서만 쓰는 로직은 함수로 빼지 않는다.** 재사용되는 것만 Layer 0/1에 둔다.

## 에이전트에게 남긴 판단 (3개)

1. **자연어 → 카카오가 먹는 검색어로 압축** — 의미론적 변환.
2. **재시도 시 어휘 교체 + `region_prefix` 지정** — '백반집'→'밥집', '관천'→'신림'.
   도로명·행정동과 사람들이 실제로 쓰는 생활권 이름의 매핑은 지역 상식이라 코드가 못 한다.
3. **`collect_info` / `collect_context` 선택** — 카페는 방문 경험(블로그)이,
   축제는 일정 사실(뉴스·웹)이 근거다. 쿼리의 의미를 이해해야 알 수 있다.

3번이 이 파이프라인을 고정 flow가 아니게 만드는 지점이다. 둘 다 쓸 수도, 하나만 쓸 수도,
결과를 보고 다른 쪽을 추가로 부를 수도 있다.

## 이전 버전 대비 변경

| 변경 | 이유 |
|---|---|
| `collect_context` 툴 신설 | 축제·관광지는 카카오만으론 일정을 알 수 없다 |
| 모든 툴 `async` 통일 | `asyncio.run()`은 주피터·서버 양쪽에서 터진다 |
| `MIN_COUNT`/`MAX_COUNT` | `TARGET_COUNT * 0.5` 비율식이 비직관적이었다 |
| `is_end` → `exhausted` 판정 | "결과가 적다"와 "더 이상 없다"는 다른 상태. 축제 쿼리의 무의미한 retry 제거 |
| `region_prefix` 인자 | 도로명 자동 추출("관천")이 실제 검색어("신림")와 어긋났다 |
| 블로그 `sim`+`date` 병합 | 매장마다 유리한 정렬이 달라 사전 판단 불가. 둘 다 던지고 dedup |
| 요약에 주소·URL 병합 | 주소 할루시네이션 제거 |
| `VERBOSE` + `_log()` | 툴 내부 진행상황만 선택적으로 확인 |

---
# Layer 0 — 공용 유틸

여러 툴에서 반복 사용되는 것만 모았다.

In [58]:
import os
import re
import json
import html
import time
import asyncio
import requests
import httpx
import functools
from pathlib import Path
from datetime import datetime
from zoneinfo import ZoneInfo
from dotenv import load_dotenv
from collections import defaultdict
load_dotenv()

TOOL_FILE_DIR = Path("./tool_file")
TOOL_FILE_DIR.mkdir(exist_ok=True)

# 노트북에서 파이프라인 내부를 들여다볼 때 True. 에이전트 실행 시엔 False 권장.
VERBOSE = True

# 동시성 제한 — 실측으로 정한 값
NAVER_SEM = asyncio.Semaphore(5)   # 네이버 API. 10 초과 시 429 빈발
SCRAPE_SEM = asyncio.Semaphore(20)  # 블로그 본문 크롤링
LLM_SEM = asyncio.Semaphore(10)     # LLM 병렬 호출

_FAIL_PREFIX = "__SCRAPE_FAIL__"    # 본문이 "[제목]" 형태로 시작하는 정상 글과 구분하기 위한 마커


def _log(*args):
    """VERBOSE일 때만 출력. 툴 내부 진행상황 확인용."""
    if VERBOSE:
        print(*args)

_STATS = defaultdict(lambda: {"n": 0, "sec": 0.0})


def _track(label: str):
    """Layer 1 함수를 감싸 호출 횟수와 누적 시간을 집계한다."""
    def deco(fn):
        if asyncio.iscoroutinefunction(fn):
            @functools.wraps(fn)
            async def wrapper(*a, **kw):
                t0 = time.perf_counter()
                try:
                    return await fn(*a, **kw)
                finally:
                    _STATS[label]["n"] += 1
                    _STATS[label]["sec"] += time.perf_counter() - t0
            return wrapper
        @functools.wraps(fn)
        def wrapper(*a, **kw):
            t0 = time.perf_counter()
            try:
                return fn(*a, **kw)
            finally:
                _STATS[label]["n"] += 1
                _STATS[label]["sec"] += time.perf_counter() - t0
        return wrapper
    return deco


def _timed_tool(fn):
    """툴 총 소요시간 + 그 안에서 쓰인 Layer 1 함수별 집계를 출력한다."""
    @functools.wraps(fn)
    async def wrapper(*a, **kw):
        _STATS.clear()
        t0 = time.perf_counter()
        try:
            return await fn(*a, **kw)
        finally:
            total = time.perf_counter() - t0
            _log(f"[{fn.__name__}] 총 {total:.1f}초")
            for label, s in sorted(_STATS.items(), key=lambda x: -x[1]["sec"]):
                _log(f"   {label:<12} {s['sec']:6.1f}초  ({s['n']}회, "
                     f"평균 {s['sec']/s['n']:.2f}초)")
    return wrapper

def dump(tool_name: str, data, run_id: str) -> str:
    """툴 결과를 파일로 떨군다. 같은 invocation의 산출물은 run_id로 묶인다."""
    path = TOOL_FILE_DIR / f"{run_id}_{tool_name}.json"
    path.write_text(json.dumps(data, ensure_ascii=False, indent=2), encoding="utf-8")
    return str(path)


def _load_json(file_path: str):
    with open(file_path, encoding="utf-8") as f:
        return json.load(f)


def _strip_tags(text: str) -> str:
    """네이버 응답의 <b> 하이라이트 태그와 HTML 엔티티 제거."""
    return html.unescape(re.sub(r"<[^>]+>", "", text or "")).strip()

In [29]:
from google.adk.tools import ToolContext


def _run_id(tool_context: ToolContext) -> str:
    """현재 invocation 식별자. 같은 요청에서 나온 파일들을 한 덩어리로 묶는다."""
    key = "temp:run_id"
    rid = tool_context.state.get(key)
    if rid is None:
        rid = str(int(time.time() * 1000))
        tool_context.state[key] = rid
    return rid

---
# Layer 1 — 외부 클라이언트

각 API를 감싸는 얇은 래퍼. 툴로 노출하지 않는다.

In [30]:
@_track("kakao")
def _kakao_keyword(query: str, page: int = 1, category_code: str = "") -> tuple[list[dict], bool]:
    """카카오 로컬 키워드 검색 1페이지. (결과, is_end) 반환.

    카카오는 상호명·카테고리·태그 문자열 매칭이라 서술형 검색어에 약하다.
    페이지당 15개, 전체 노출 가능 문서는 45개가 상한.
    """
    params = {"query": query, "size": 15, "page": page}
    if category_code:
        params["category_group_code"] = category_code

    resp = requests.get(
        "https://dapi.kakao.com/v2/local/search/keyword.json",
        headers={"Authorization": f"KakaoAK {os.environ['KAKAO_REST_API_KEY']}"},
        params=params,
        timeout=5,
    )
    resp.raise_for_status()
    body = resp.json()

    places = [
        {
            "id": d["id"],
            "name": d["place_name"],
            "category": d["category_name"],
            "address": d["road_address_name"] or d["address_name"],
            "phone": d.get("phone", ""),
            "url": d["place_url"],
            "source": "kakao_local",
        }
        for d in body["documents"]
    ]
    return places, body["meta"]["is_end"]

In [31]:
@_track("naver")
async def _naver_search(query: str, kind: str = "blog", display: int = 10,
                        sort: str = "sim") -> tuple[list[dict], str]:
    """네이버 검색 API 비동기 호출. blog / news / webkr 모두 같은 응답 구조라 하나로 처리.

    kind: 'blog'(블로그) / 'news'(뉴스) / 'webkr'(웹문서)
    sort: 'sim'(정확도) / 'date'(최신). webkr은 sort 미지원이라 무시된다.
    반환: (결과 리스트, 상태) — 상태는 'ok' / 'empty' / '429'
    """
    async with NAVER_SEM:
        for attempt in range(3):
            async with httpx.AsyncClient(timeout=5) as client:
                resp = await client.get(
                    f"https://openapi.naver.com/v1/search/{kind}",
                    headers={
                        "X-Naver-Client-Id": os.environ["NAVER_CLIENT_ID"],
                        "X-Naver-Client-Secret": os.environ["NAVER_CLIENT_SECRET"],
                    },
                    params={"query": query, "display": display, "sort": sort},
                )
            if resp.status_code == 429:
                await asyncio.sleep(0.5 * (attempt + 1))
                continue
            resp.raise_for_status()
            break
        else:
            return [], "429"

    items = resp.json().get("items", [])
    if not items:
        return [], "empty"

    # 필드명이 종류별로 다르다: 블로그는 postdate, 뉴스는 pubDate, 웹문서는 날짜 없음
    return [
        {
            "title": _strip_tags(i.get("title", "")),
            "description": _strip_tags(i.get("description", "")),
            "link": i.get("link", ""),
            "date": i.get("postdate", "") or i.get("pubDate", ""),
        }
        for i in items
    ], "ok"

In [32]:
_BODY_SELECTORS = [
    "div.se-main-container",   # 네이버 스마트에디터 ONE
    "#postViewArea",           # 네이버 구버전 에디터
    "div#post-view",
    "article",                 # 일반 웹페이지
    "main",
]


async def scrape_urls(urls: list[str], max_chars: int = 0) -> list[dict]:
    """URL 병렬 크롤링 → [{'url', 'text'}]. 실패해도 예외 대신 마커 문자열을 담는다.

    실패 판정은 text.startswith(_FAIL_PREFIX)로 한다.
    max_chars가 0이면 본문 전체를 담는다.
    """
    from bs4 import BeautifulSoup

    ua = ("Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
          "(KHTML, like Gecko) Chrome/120.0 Safari/537.36")
    @_track("scrape")
    async def fetch_one(client, url: str) -> str:
        # 네이버 블로그는 iframe 껍데기라 PostView로 우회해야 본문이 나온다
        m = re.search(r"blog\.naver\.com/([^/?]+)/(\d+)", url)
        target = (f"https://blog.naver.com/PostView.naver?blogId={m.group(1)}&logNo={m.group(2)}"
                  if m else url)

        async with SCRAPE_SEM:
            try:
                resp = await client.get(target)
                resp.raise_for_status()
            except httpx.HTTPError as e:
                return f"{_FAIL_PREFIX}열기 실패: {type(e).__name__}"

        soup = BeautifulSoup(resp.text, "html.parser")
        for tag in soup(["script", "style", "nav", "footer"]):
            tag.decompose()

        # 셀렉터를 순회하되, 잡히긴 했는데 내용이 짧으면(광고 div 등) 다음 것으로 넘어간다
        text = ""
        for selector in _BODY_SELECTORS:
            node = soup.select_one(selector)
            if node:
                candidate = re.sub(r"\s+", " ", node.get_text(separator=" ", strip=True)).strip()
                if len(candidate) > 150:
                    text = candidate
                    break
        if not text:
            text = re.sub(r"\s+", " ", (soup.body or soup).get_text(separator=" ", strip=True)).strip()

        if len(text) < 150:
            return f"{_FAIL_PREFIX}본문 추출 실패"
        return text[:max_chars] if max_chars else text

    async with httpx.AsyncClient(
        headers={"User-Agent": ua}, timeout=10.0, follow_redirects=True
    ) as client:
        texts = await asyncio.gather(*(fetch_one(client, u) for u in urls))

    return [{"url": u, "text": t} for u, t in zip(urls, texts)]

In [33]:
from google import genai
from google.genai import types

_client = genai.Client(api_key=os.environ["GOOGLE_API_KEY"])
UTIL_MODEL = "gemini-3.1-flash-lite"   # 툴 내부 유틸용. 에이전트 모델과 별개로 교체 가능


def _llm(prompt: str, *, json_mode: bool = False, temperature: float = 0.0) -> str:
    """유틸 LLM 동기 호출. 단발 처리용."""
    config = types.GenerateContentConfig(temperature=temperature)
    if json_mode:
        config.response_mime_type = "application/json"
    resp = _client.models.generate_content(model=UTIL_MODEL, contents=prompt, config=config)
    return resp.text

@_track("llm")
async def _llm_async(prompt: str, *, json_mode: bool = False, temperature: float = 0.0) -> str:
    """유틸 LLM 비동기 호출. 병렬 처리용."""
    config = types.GenerateContentConfig(temperature=temperature)
    if json_mode:
        config.response_mime_type = "application/json"
    resp = await _client.aio.models.generate_content(
        model=UTIL_MODEL, contents=prompt, config=config
    )
    return resp.text

---
# Layer 2 — 툴

에이전트에 노출되는 함수들. docstring이 곧 LLM에게 주는 명세다.

In [34]:
def get_today() -> dict:
    """오늘 날짜와 요일, 현재 시각을 알려준다.

    '오늘', '이번 주말', '지금 하는' 같은 상대적 시점이 요청에 포함되거나,
    축제·행사처럼 기간이 있는 대상을 찾을 때 먼저 호출해 현재 시점을 확인한다.

    Returns:
        {"date": "2026-08-13", "weekday": "목요일", "time": "14:30", "year": 2026, "month": 8}
    """
    now = datetime.now(ZoneInfo("Asia/Seoul"))
    weekdays = ["월요일", "화요일", "수요일", "목요일", "금요일", "토요일", "일요일"]
    return {
        "date": now.strftime("%Y-%m-%d"),
        "weekday": weekdays[now.weekday()],
        "time": now.strftime("%H:%M"),
        "year": now.year,
        "month": now.month,
    }

In [ ]:
_EXTRACT_NAMES_PROMPT = """\
아래는 네이버 블로그 검색 결과다. 각 줄은 '제목 | 요약' 형식이다.
여기서 실제 존재하는 가게/장소의 고유 상호명만 뽑아라.
- 제목에 상호명이 있을 확률이 높다. 요약은 보조로만 참고한다.
- '카페 추천', '맛집 베스트10', '데이트코스' 같은 일반명사구는 상호명이 아니다.
- 같은 매장이 여러 줄에 걸쳐 나오면 한 번만 적어라.
- 확실하지 않으면 넣지 마라.
- 결과는 문자열 배열 JSON으로만 출력한다. 예: ["블루보틀 성수", "어니언 성수"]
"""

MIN_COUNT = 20      # 이 미만이면 부족하다고 보고 재시도/폴백
MAX_COUNT = 30      # 이만큼 모이면 수집 종료
MAX_ATTEMPTS = 3

@_timed_tool
async def collect_places(
    query: str,
    tool_context: ToolContext,
    category_code: str = "",
) -> dict:
    """지역 키워드로 장소 후보 목록을 수집한다. 모든 작업의 첫 단계다.

    카카오 로컬 검색 기반이라 이름·주소·좌표는 정확하지만, 영업시간이나 행사 일정처럼
    변하는 정보는 담기지 않는다. 그런 정보가 필요하면 이 툴 다음에 collect_context를 쓴다.

    검색어는 '지역 + 짧은 명사' 형태여야 한다. 서술형 문장은 결과가 거의 나오지 않는다.
      좋은 예: '성수동 카페', '진해 군항제', '가평 캠핑장'
      나쁜 예: '성수동에서 조용히 작업하기 좋은 카페'

    Args:
        query: 검색어. '지역 + 짧은 명사' 조합.
        category_code: 카테고리 필터. 'CE7'=카페, 'FD6'=음식점, 'AT4'=관광명소, 'CT1'=문화시설.
                       확실하지 않으면 빈 문자열로 두는 편이 결과가 많다.

    Returns:
        status가 'retry'면 결과가 부족한 것이므로, message 안내대로 query를 바꿔 다시 호출한다.
        status가 'ok'면 수집이 끝난 것이므로 다시 호출하지 말고 다음 단계로 넘어간다.
        축제나 특정 시설처럼 대상이 하나뿐인 경우 결과가 몇 개여도 'ok'가 나온다. 정상이다.
        예시(완료): {'status': 'ok', 'count': 40, 'file': './tool_file/..._collect_places.json'}
    """
    attempt_key = "temp:collect_attempts"
    attempts = tool_context.state.get(attempt_key, 0) + 1
    tool_context.state[attempt_key] = attempts

    places: dict[str, dict] = {}

    # 1) 카카오 로컬 페이지네이션 (id 기준 dedup)
    #    is_end로 "카카오가 가진 걸 다 줬는지"를 구분한다.
    #    결과가 적은 것과 더 이상 없는 것은 완전히 다른 상태다.
    page = 1
    exhausted = False
    while len(places) < MAX_COUNT and page <= 45:
        batch, is_end = _kakao_keyword(query, page=page, category_code=category_code)
        for p in batch:
            places.setdefault(p["id"], p)
        if is_end:
            exhausted = True
            break
        page += 1
    _log(f"[collect_places] '{query}' 카카오 {len(places)}개 "
         f"(시도 {attempts}/{MAX_ATTEMPTS}, exhausted={exhausted})")

    # 2) 부족한데 더 나올 여지가 있으면 → 에이전트에게 쿼리 재구성 요청
    #    같은 쿼리 재시도는 같은 결과이므로, 어휘를 바꾸는 판단은 LLM 몫이다.
    #    exhausted면 무엇을 해도 결과가 같으므로 재시도하지 않는다.
    if len(places) < MIN_COUNT and attempts < MAX_ATTEMPTS and not exhausted:
        return {
            "status": "retry",
            "message": (
                f"'{query}' 검색 결과가 {len(places)}개로 부족합니다(최소 {MIN_COUNT}개 필요). "
                f"({attempts}/{MAX_ATTEMPTS}번째 시도) "
                "지역명은 유지하고 업종어를 카카오에 등록될 법한 다른 표현으로 바꿔 다시 호출하세요. "
                "예: '백반집' → '밥집' 또는 '한식', '작업 카페' → '카공카페' 또는 '스터디카페'"
            ),
        }

    # 3) 재시도를 다 쓰고도 부족하면 → 블로그에서 상호명을 뽑아 카카오로 실존 검증 후 보충
    #    exhausted면 억지로 채우지 않는다. 축제 검색에 딸려온 주변 식당이 섞이기 때문.
    if len(places) < MIN_COUNT and not exhausted:
        blog_items, _ = await _naver_search(query, kind="blog", display=30, sort="sim")
        lines = "\n".join(f"{i+1}. {b['title']} | {b['description']}"
                           for i, b in enumerate(blog_items))
        try:
            names = json.loads(_llm(f"{_EXTRACT_NAMES_PROMPT}\n{lines}", json_mode=True))
            names = [n.strip() for n in names if isinstance(n, str) and n.strip()]
        except (json.JSONDecodeError, TypeError):
            names = []
        _log(f"[collect_places] 블로그 폴백 → 후보명 {len(names)}개")

        for name in names:
            if len(places) >= MAX_COUNT:
                break
            verified, _ = _kakao_keyword(name, page=1)
            for p in verified:                       # 실존하는 것만, 이름당 1건
                if p["id"] not in places:
                    p["source"] = "blog_derived"
                    places[p["id"]] = p
                    break

    result = list(places.values())[:MAX_COUNT]
    file_path = dump("collect_places", result, _run_id(tool_context))
    tool_context.state["temp:last_places_file"] = file_path

    return {
        "status": "ok",
        "count": len(result),
        "source_breakdown": {
            "local": sum(1 for p in result if p["source"] == "kakao_local"),
            "blog_derived": sum(1 for p in result if p["source"] == "blog_derived"),
        },
        "file": file_path,
    }

In [36]:
_SUMMARIZE_PROMPT = """\
사용자 요청: "{user_query}"

아래는 '{place_name}'에 대한 블로그 후기 {n}건이다.
여러 후기를 종합해 위 요청 관점에서 이 장소를 평가하고 다음 JSON 형식으로만 답해라.

{{
  "strengths": ["요청 조건에 부합하는 점 2~4개"],
  "weaknesses": ["요청 조건에 아쉬운 점 또는 주의할 점 0~3개"],
  "summary": "전반적인 평가 2~3문장"
}}

규칙:
- 여러 후기에서 반복되는 내용일수록 신뢰도가 높다. 한 후기에만 나오는 개인적 감상은 비중을 낮춰라.
- 후기 간 내용이 상충하면 그 사실을 그대로 적어라. (예: "시간대에 따라 웨이팅 편차가 큼")
- 본문에 실제로 언급된 내용만 사용하고 추측하지 마라.
- 홍보성 과장 표현은 걷어내고 실제 경험 위주로 판단해라.
- 후기가 이 장소에 대한 내용이 아니거나 정보가 부실하면
  {{"strengths": [], "weaknesses": [], "summary": "정보 부족"}}로 답해라.
"""

@_timed_tool
async def collect_info(tool_context: ToolContext, user_query: str, region_prefix: str) -> dict:
    """수집된 장소들의 블로그 후기를 모아 장점·단점 중심으로 요약한다.

    실제 방문자 경험(맛, 분위기, 혼잡도, 1인석 유무 등)이 판단 근거로 필요할 때 쓴다.
    카페·음식점처럼 방문 후기가 많은 대상에 적합하다.
    축제·행사처럼 '언제 열리는지'가 중요한 대상은 collect_context를 대신 쓴다.

    Args:
        user_query: 사용자의 원래 요청 문장. 어떤 관점에서 장단점을 판단할지 결정한다.
                    예: '신림역 혼밥하기 좋은 백반집'
        region_prefix: 블로그 검색어에 붙일 지역명. 사람들이 실제로 그 동네를 부르는
                    통칭을 쓴다. 도로명이나 행정동명이 아니다.
                    예: 관악구 관천로 일대 → '신림', 금천구 가산디지털2로 → '가산'

    Returns:
        {"count": int, "no_info": int, "url_file": str, "summary_file": str}
        count는 요약이 만들어진 장소 수, no_info는 후기를 못 찾은 장소 수.
        판단 근거는 summary_file에 있다. read_file로 읽어라.
    """
    places_file = tool_context.state.get("temp:last_places_file")
    if not places_file:
        return {"status": "error",
                "message": "collect_places를 먼저 호출하세요. 이 툴을 다시 호출해도 같은 결과입니다."}

    places = _load_json(places_file)
    run_id = _run_id(tool_context)

    # --- 1) 매장별 블로그 URL 수집 ---
    # 도로명은 행정 표기라 실제 리뷰에서 쓰는 생활권 이름과 다르다("관천" vs "신림").
    # 그 매핑은 지역 상식이라 코드가 못 하므로 에이전트가 준 region_prefix를 쓴다.
    # sim/date는 매장마다 유리한 쪽이 달라 사전 판단이 불가능하므로 둘 다 던지고 합친다.
    queries = [f"{region_prefix} {p['name']}".strip() for p in places]

    raw = await asyncio.gather(*[
        _naver_search(q, kind="blog", display=10, sort=s)
        for q in queries for s in ("sim", "date")
    ])

    urls, reasons = {}, {}
    for i, p in enumerate(places):
        merged = raw[i * 2][0] + raw[i * 2 + 1][0]
        statuses = (raw[i * 2][1], raw[i * 2 + 1][1])

        seen, hits = set(), []
        for b in merged:
            if b["link"] in seen:
                continue
            seen.add(b["link"])
            # 검색어가 통해도 무관한 글이 섞이므로 상호명 실제 등장 여부로 한 번 더 거른다
            if p["name"] in b["title"] or p["name"] in b["description"]:
                hits.append(b)

        urls[p["name"]] = hits[:5]
        if not hits:
            reasons[p["name"]] = ("empty" if all(s == "empty" for s in statuses)
                                  else "no_match_after_filter")

    url_file = dump("info_urls", urls, run_id)
    if reasons:
        from collections import Counter
        _log(f"[collect_info] 후기 미확보 {len(reasons)}개 → {dict(Counter(reasons.values()))}")

    # --- 2) 본문 크롤링 (파일로 남기지 않음: 요약이 나오면 원문은 용도 끝) ---
    links, owner = [], []
    for name, items in urls.items():
        for idx, item in enumerate(items):
            links.append(item["link"])
            owner.append((name, idx))

    scraped = await scrape_urls(links)
    bodies: dict[str, list[str]] = {name: [] for name in urls}
    for (name, _), s in zip(owner, scraped):
        if not s["text"].startswith(_FAIL_PREFIX):
            bodies[name].append(s["text"])
    _log(f"[collect_info] 크롤링 {len(scraped)}건 중 성공 {sum(len(v) for v in bodies.values())}건")

    # --- 3) 매장당 후기 전체를 한 번에 넣어 통합 요약 ---
    # 후기를 개별 요약하면 반복 빈도(=신뢰도) 정보가 사라진다.
    async def summarize_one(name: str, texts: list[str]) -> dict:
        joined = "\n\n---\n\n".join(f"[후기 {i+1}]\n{t[:2000]}" for i, t in enumerate(texts))
        prompt = _SUMMARIZE_PROMPT.format(
            user_query=user_query, place_name=name, n=len(texts)) + "\n" + joined
        async with LLM_SEM:
            raw_out = await _llm_async(prompt, json_mode=True)
        try:
            return json.loads(raw_out)
        except json.JSONDecodeError:
            return {"strengths": [], "weaknesses": [], "summary": "정보 부족"}

    names = [n for n, t in bodies.items() if t]
    results = await asyncio.gather(*[summarize_one(n, bodies[n]) for n in names])

    # 주소·URL을 요약에 병합 — 이게 없으면 최종 답변에서 주소를 지어낸다
    meta = {p["name"]: p for p in places}
    summaries = {
        n: {
            **r,
            "review_count": len(bodies[n]),
            "address": meta[n].get("address", ""),
            "category": meta[n].get("category", ""),
            "url": meta[n].get("url", ""),
        }
        for n, r in zip(names, results)
    }
    summary_file = dump("info_summary", summaries, run_id)

    return {
        "count": len(summaries),
        "no_info": len(places) - len(summaries),
        "url_file": url_file,
        "summary_file": summary_file,
    }

In [37]:
_CONTEXT_PROMPT = """\
사용자 요청: "{user_query}"
오늘 날짜: {today}

아래는 '{name}'에 대한 웹문서·뉴스 {n}건이다.
사용자 요청에 답하는 데 필요한 사실 정보만 추려 다음 JSON 형식으로만 답해라.

{{
  "period": "행사 기간이나 운영 시기. 본문에 없으면 빈 문자열",
  "facts": ["확인된 사실 2~4개 (내용, 규모, 요금, 예약 방법 등)"],
  "cautions": ["방문 전 알아야 할 점 0~3개"],
  "summary": "전반적인 설명 2~3문장",
  "freshness": "본문 정보가 올해 기준이면 'current', 지난 연도 기준이면 'outdated', 판단 불가면 'unknown'"
}}

규칙:
- 본문에 실제로 적힌 내용만 사용하고 추측하지 마라.
- 날짜가 나오면 연도를 반드시 확인해라. 오늘 날짜 기준으로 지난 정보면 freshness를 'outdated'로 표기해라.
- 광고성 문구와 목록형 나열 글은 신뢰도를 낮게 봐라.
- 관련 정보가 없으면 {{"period": "", "facts": [], "cautions": [], "summary": "정보 부족", "freshness": "unknown"}}로 답해라.
"""

@_timed_tool
async def collect_context(tool_context: ToolContext, user_query: str, top_n: int = 10) -> dict:
    """수집된 장소들의 일정·운영정보를 웹문서와 뉴스에서 찾아 정리한다.

    축제·행사·전시처럼 '언제 열리는지'가 핵심이거나, 관광지·캠핑장처럼 예약·요금·시설
    정보가 필요할 때 쓴다. 뉴스는 날짜와 장소가 사실로 명시되어 블로그보다 신뢰도가 높다.

    카페·음식점처럼 방문자 경험이 중요한 대상은 collect_info를 대신 쓴다.

    Args:
        user_query: 사용자의 원래 요청 문장.
        top_n: 상위 몇 개 장소를 조사할지. 축제·행사는 후보가 적으므로 기본값으로 충분하다.

    Returns:
        {"count": int, "no_info": int, "source_file": str, "context_file": str}
        정리된 사실 정보는 context_file에 있다. read_file로 읽어라.
        freshness가 'outdated'인 항목은 지난 연도 정보이므로 주의해서 다뤄라.
    """
    places_file = tool_context.state.get("temp:last_places_file")
    if not places_file:
        return {"status": "error",
                "message": "collect_places를 먼저 호출하세요. 이 툴을 다시 호출해도 같은 결과입니다."}

    places = _load_json(places_file)[:top_n]
    run_id = _run_id(tool_context)
    today = get_today()

    # --- 1) 웹문서 + 뉴스 동시 검색 ---
    # 웹문서는 공식 페이지·관공서 자료에, 뉴스는 일정·규모 사실에 강하다. 둘을 합쳐야 커버가 된다.
    tasks = []
    for p in places:
        region = p.get("address", "").split()[0] if p.get("address") else ""
        tasks.append(_naver_search(f"{region} {p['name']}".strip(), kind="webkr", display=5))
        tasks.append(_naver_search(f"{p['name']} {today['year']}", kind="news", display=5, sort="date"))
    raw = await asyncio.gather(*tasks)

    sources = {}
    for i, p in enumerate(places):
        web_items, _ = raw[i * 2]
        news_items, _ = raw[i * 2 + 1]
        merged = ([{**w, "kind": "web"} for w in web_items]
                  + [{**n, "kind": "news"} for n in news_items])
        # 이름이 실제로 언급된 문서만 채택
        sources[p["name"]] = [d for d in merged
                              if p["name"] in d["title"] or p["name"] in d["description"]]
    source_file = dump("context_sources", sources, run_id)
    _log(f"[collect_context] {len(places)}곳 중 문서 확보 "
         f"{sum(1 for v in sources.values() if v)}곳")

    # --- 2) 본문 크롤링 ---
    links, owner = [], []
    for name, items in sources.items():
        for item in items[:4]:
            links.append(item["link"])
            owner.append(name)

    scraped = await scrape_urls(links, max_chars=4000)
    bodies: dict[str, list[str]] = {name: [] for name in sources}
    for name, s in zip(owner, scraped):
        if not s["text"].startswith(_FAIL_PREFIX):
            bodies[name].append(s["text"])

    # 본문 크롤링이 다 실패해도 검색 스니펫만으로 요약을 시도한다
    # (공식 페이지는 봇 차단이 잦은데, 스니펫에 일정이 그대로 담긴 경우가 많다)
    for name, items in sources.items():
        if not bodies[name] and items:
            bodies[name] = [
                "\n".join(f"{d['title']} | {d['description']} ({d['date']})" for d in items)
            ]
    _log(f"[collect_context] 크롤링 {len(scraped)}건 중 성공 "
         f"{sum(1 for s in scraped if not s['text'].startswith(_FAIL_PREFIX))}건")

    # --- 3) 장소별 사실 정리 ---
    async def summarize_one(name: str, texts: list[str]) -> dict:
        joined = "\n\n---\n\n".join(f"[자료 {i+1}]\n{t}" for i, t in enumerate(texts))
        prompt = _CONTEXT_PROMPT.format(
            user_query=user_query, today=today["date"], name=name, n=len(texts)) + "\n" + joined
        async with LLM_SEM:
            raw_out = await _llm_async(prompt, json_mode=True)
        try:
            return json.loads(raw_out)
        except json.JSONDecodeError:
            return {"period": "", "facts": [], "cautions": [],
                    "summary": "정보 부족", "freshness": "unknown"}

    names = [n for n, t in bodies.items() if t]
    results = await asyncio.gather(*[summarize_one(n, bodies[n]) for n in names])

    meta = {p["name"]: p for p in places}
    contexts = {
        n: {
            **r,
            "source_count": len(bodies[n]),
            "address": meta[n].get("address", ""),
            "category": meta[n].get("category", ""),
            "url": meta[n].get("url", ""),
        }
        for n, r in zip(names, results)
    }
    context_file = dump("context_summary", contexts, run_id)

    return {
        "count": len(contexts),
        "no_info": len(places) - len(contexts),
        "source_file": source_file,
        "context_file": context_file,
    }

In [38]:
def read_file(file_path: str) -> dict:
    """툴이 만든 결과 파일을 읽는다. 최종 답변을 쓰기 전에 반드시 호출한다.

    collect_info의 summary_file, collect_context의 context_file에 판단 근거가 들어있다.
    파일을 읽지 않고 답변을 지어내면 안 된다.

    Args:
        file_path: 이전 툴이 반환한 file / summary_file / context_file 값.

    Returns:
        {"content": ...} 파일 내용.
    """
    path = Path(file_path)
    if not path.exists():
        return {"status": "error", "message": f"파일을 찾을 수 없습니다: {file_path}"}
    if not str(path.resolve()).startswith(str(TOOL_FILE_DIR.resolve())):
        return {"status": "error", "message": "허용되지 않은 경로입니다."}
    try:
        return {"content": _load_json(str(path))}
    except json.JSONDecodeError:
        return {"status": "error", "message": "파일을 읽을 수 없습니다."}

---
# Layer 3 — 에이전트

## 에이전트에게 남긴 판단

코드로 내릴 수 있는 것(페이지네이션, dedup, 폴백 조건, 도로명 추출, 동시성 제어)은 전부
함수 안으로 내렸다. 에이전트가 실제로 판단하는 건 셋이다.

1. **자연어 → 카카오가 먹는 검색어로 압축** — 의미론적 변환이라 코드가 못 한다.
2. **재시도 시 어휘를 바꾸는 것** — '백반집' → '밥집'. 동의어 판단.
3. **collect_info / collect_context 중 무엇을 쓸지** — 이번 버전의 핵심.
   같은 "장소 찾기"라도 카페는 방문 경험(블로그)이, 축제는 일정 사실(뉴스·웹)이 필요하다.
   이건 쿼리의 의미를 이해해야 알 수 있으므로 코드가 관측할 수 없다.

3번이 이 파이프라인을 선형 flow가 아니게 만드는 지점이다. 둘 다 쓸 수도, 하나만 쓸 수도,
결과를 보고 다른 쪽을 추가로 부를 수도 있다.

In [39]:
from pydantic import BaseModel, Field
from typing import List


class PlaceCard(BaseModel):
    name: str = Field(description="장소명")
    address: str = Field(description="주소. 파일에 있는 값을 그대로 쓴다")
    category: str = Field(default="", description="카테고리")
    reason: str = Field(description="이 요청에 왜 맞는지 2~3문장")
    highlights: List[str] = Field(description="핵심 강점 2~3개. 짧은 구문으로")
    caution: str = Field(default="", description="주의사항. 없으면 빈 문자열")
    period: str = Field(default="", description="행사 기간. 해당 없으면 빈 문자열")
    url: str = Field(default="", description="카카오맵 상세 페이지 URL")


class FeedOutput(BaseModel):
    places: List[PlaceCard] = Field(description="추천 장소 5~8곳")
    note: str = Field(default="", description="전체에 대한 참고사항. 없으면 빈 문자열")

In [40]:
from google.adk.agents import LlmAgent
from google.adk.runners import InMemoryRunner

APP_NAME = "wh_poc"
USER_ID = "user1"
AGENT_MODEL = "gemini-3.1-flash-lite"

INSTRUCTION = """당신은 지역 장소를 찾아 추천하는 어시스턴트입니다.

## 1단계 — 후보 수집
사용자 요청에서 '지역 + 짧은 명사' 형태의 검색어를 만들어 collect_places를 호출합니다.
  예: "성수동에서 조용히 작업할 카페" → "성수동 카공카페"
  예: "이번 봄에 진해 벚꽃 축제 언제야" → "진해 군항제"

status가 'retry'면 message 안내대로 검색어의 업종어를 다른 표현으로 바꿔 즉시 다시 호출하세요.
사용자에게 되묻지 말고 스스로 조정합니다.
축제나 특정 시설처럼 대상이 하나뿐이면 결과가 몇 개여도 'ok'가 나옵니다. 정상이니 다음 단계로 갑니다.

## 2단계 — 어떤 정보가 필요한지 판단
요청의 성격에 따라 다음 중 하나 또는 둘 다를 선택합니다. 이 판단이 답변 품질을 좌우합니다.

- **collect_info** (블로그 후기 요약)
  방문자의 실제 경험이 판단 근거인 경우. 맛, 분위기, 혼잡도, 좌석, 서비스 등.
  카페·음식점·술집 등 상설 매장 추천에 적합합니다.

  region_prefix에는 그 동네를 사람들이 실제로 부르는 이름을 넣으세요.
  주소의 도로명이나 행정동을 그대로 쓰면 블로그 검색이 실패합니다.
    예: 사용자가 "신림역"이라 했고 주소가 "관악구 관천로"면 → '신림'
    예: 주소가 "금천구 가산디지털2로"면 → '가산'

- **collect_context** (웹문서·뉴스 사실 정리)
  일정·기간·요금·예약처럼 확인된 사실이 필요한 경우.
  축제·행사·전시처럼 열리는 시기가 중요하거나, 관광지·캠핑장처럼 운영 정보가 필요할 때 씁니다.
  시점이 걸린 요청이면 먼저 get_today로 오늘 날짜를 확인하세요.

애매하면 둘 다 호출해도 됩니다. 예를 들어 "캠핑장 추천"은 운영 정보(collect_context)와
이용 후기(collect_info)가 모두 도움이 됩니다.

## 3단계 — 근거 확인
read_file로 summary_file 또는 context_file을 읽습니다.
파일을 읽지 않고 답변을 만들면 안 됩니다.

## 4단계 — 추천
사용자 요청에 가장 잘 맞는 5~8곳을 골라 JSON으로 응답합니다.
- review_count 또는 source_count가 많을수록 신뢰도가 높습니다.
- freshness가 'outdated'인 정보는 지난 연도 것이므로 caution에 그 사실을 적으세요.
- address와 url은 파일에 있는 값을 그대로 쓰세요. 절대 지어내지 마세요.
- 파일에 없는 정보는 쓰지 마세요. 강점이 부족하면 그 장소를 빼는 편이 낫습니다.

## 실패 처리
collect_info나 collect_context가 error를 반환하면 collect_places부터 다시 실행하세요.
정보 수집이 실패했는데 collect_places 결과만으로 평가를 지어내면 안 됩니다.
"""

agent = LlmAgent(
    model=AGENT_MODEL,
    name="place_finder",
    description="지역 기반 장소 추천 에이전트",
    instruction=INSTRUCTION,
    tools=[get_today, collect_places, collect_info, collect_context, read_file],
    output_schema=FeedOutput,
)

runner = InMemoryRunner(agent=agent, app_name=APP_NAME)

In [41]:
import uuid


async def run(query: str, show_tools: bool = True) -> dict:
    """에이전트 실행. 이벤트 스트림을 파싱해 전체 결과를 dict로 반환한다."""
    session_id = str(uuid.uuid4())
    await runner.session_service.create_session(
        app_name=APP_NAME, user_id=USER_ID, session_id=session_id
    )
    content = types.Content(role="user", parts=[types.Part(text=query)])

    out = {"query": query, "session_id": session_id, "invocation_id": None,
           "tool_calls": [], "files": {}, "final_text": ""}

    async for event in runner.run_async(
        user_id=USER_ID, session_id=session_id, new_message=content
    ):
        if out["invocation_id"] is None:
            out["invocation_id"] = event.invocation_id

        for call in event.get_function_calls():
            out["tool_calls"].append({"name": call.name, "args": dict(call.args)})
            if show_tools:
                print(f"[CALL] {call.name}({dict(call.args)})")

        for fr in event.get_function_responses():
            r = fr.response
            if isinstance(r, dict):
                for k in ("file", "url_file", "summary_file", "source_file", "context_file"):
                    if k in r:
                        out["files"][k] = r[k]
                if show_tools:
                    brief = {k: v for k, v in r.items() if k != "content"}
                    if "content" in r:
                        brief["content"] = f"<{len(str(r['content']))}자>"
                    print(f"[RESP] {fr.name} → {brief}")

        if event.is_final_response() and event.content and event.content.parts:
            out["final_text"] = "".join(p.text for p in event.content.parts if p.text).strip()

    try:
        out["result"] = json.loads(out["final_text"])
    except json.JSONDecodeError:
        out["result"] = None
    return out

---
# 테스트

각 계층을 독립적으로 확인한다. 위에서부터 순서대로 실행하면 앞 단계 결과를 뒷 단계가 쓴다.

## Layer 1 — 클라이언트 단위 확인

In [42]:
# 카카오: 상설 매장 vs 축제. 축제는 결과가 적게 나오는 게 정상이다.
for q in ["신림역 밥집", "진해 군항제"]:
    places, is_end = _kakao_keyword(q)
    print(f"{q}: {len(places)}개 (is_end={is_end})")
    for p in places[:3]:
        print(f"   {p['name']} | {p['category']} | {p['address']}")

신림역 밥집: 15개 (is_end=False)
   신림춘천집 본점 | 음식점 > 한식 > 육류,고기 > 닭요리 | 서울 관악구 남부순환로 1600-8
   원조민속순대타운 | 음식점 > 한식 > 순대 | 서울 관악구 신림로59길 14
   막불감동 | 음식점 > 한식 > 국수 | 서울 관악구 남부순환로 1599
진해 군항제: 6개 (is_end=True)
   진해군항제 | 이슈 > 이벤트 > 페스티벌 | 경남 창원시 진해구 평안동 10
   여좌천 | 여행 > 관광,명소 > 하천 | 경남 창원시 진해구 여좌동
   이충무공선양군항제위원회 | 사회,공공기관 > 단체,협회 | 경남 창원시 진해구 백구로 41-1


In [43]:
# 네이버 3종 비교. 같은 쿼리에 소스별로 뭐가 나오는지 눈으로 본다.
for kind in ["blog", "news", "webkr"]:
    items, status = await _naver_search("진해 군항제", kind=kind, display=3, sort="date")
    print(f"\n--- {kind} ({status}) ---")
    for i in items:
        print(f"  {i['title'][:60]}  [{i['date'][:10]}]")


--- blog (ok) ---
  진해군항제, 한국 대표 축제 평가 `종합 2위` 쾌거:창원일보  [20260813]
  진해군항제, 한국 대표 축제 평가 종합 2위 쾌거  [20260812]
  진해 자영업자대출 당일일수 정보 먼저 쉽게 받아보기  [20260812]

--- news (ok) ---
  진해군항제, 전국 축제 ‘TOP2’  [Thu, 13 Au]
  해군사관학교 통합·이전 추진에 들끓는 창원 진해[현장]  [Thu, 13 Au]
  [창원 소식] 강기윤 시장 이틀 연속 국회 방문 국비 확보 총력 등  [Thu, 13 Au]

--- webkr (ok) ---
  분야별정보 - 창원특례시청  []
  진해 군항제 - 나무위키  []
  진해군항제 | 문화관광축제 | 대한민국 구석구석 축제  []


In [44]:
# 크롤링: 실패는 예외가 아니라 마커 문자열로 돌아온다.
items, _ = await _naver_search("봉천 도화불백", kind="blog", display=3, sort="date")
scraped = await scrape_urls([i["link"] for i in items])
for s in scraped:
    ok = not s["text"].startswith(_FAIL_PREFIX)
    print(f"[{'OK ' if ok else 'FAIL'}] {len(s['text'])}자 | {s['text'][:80]}")

[OK ] 1076자 | 도화불백 신림본점 서울특별시 관악구 봉천로 304 1층 도화불백 신림본점 오늘은 고기 한 점에 밥 한 숟가락 딱 좋은 곳, 도화불백에 다녀왔습니
[OK ] 595자 | 어제는 #도화불백 #도화불백봉천 에 갔다.. 직원의 추천으로 가본 불백이랑 제육의 맛집이라고한다~! ​ 입장은 2시에 했으며 시간대가 시간대인만
[OK ] 1374자 | ​ 신림 봉천 맛집 도화불백 ​ ⏰ 영업시간 매일 11:30 ~ 21:00 (라스트 오더 20:30) 첫째 주, 셋째 주 토요일 휴무 ​ 🚗 주


In [45]:
# LLM 병렬 동작 확인. 5개가 1개 호출 시간과 비슷하면 정상.
start = time.time()
await asyncio.gather(*[_llm_async("1+1은? 숫자만") for _ in range(5)])
print(f"5개 병렬: {time.time() - start:.1f}초")

5개 병렬: 1.0초


## Layer 2 — 툴 단위 확인

툴은 `tool_context`를 받으므로 노트북에서 직접 부르려면 대역이 필요하다.

In [46]:
class FakeToolContext:
    """ADK가 주입하는 ToolContext 대역. 같은 인스턴스를 재사용하면 state가 유지되어
    재시도 카운트나 파일 경로 전달을 그대로 재현할 수 있다."""
    def __init__(self):
        self.state = {}


ctx = FakeToolContext()
print(get_today())

{'date': '2026-08-13', 'weekday': '목요일', 'time': '20:33', 'year': 2026, 'month': 8}


In [47]:
# collect_places — 결과가 부족하면 retry, 카카오가 다 소진되면(exhausted) 바로 ok.
r = await collect_places("신림역 밥집", ctx)
print(r)

[collect_places] '신림역 밥집' 카카오 45개 (시도 1/3, exhausted=True)
[collect_places] 총 0.3초
   kakao           0.3초  (3회, 평균 0.10초)
{'status': 'ok', 'count': 40, 'source_breakdown': {'local': 40, 'blog_derived': 0}, 'file': 'tool_file\\1786620787035_collect_places.json'}


In [48]:
# retry 흐름 재현: 일부러 안 잡히는 검색어로 시작해 어휘를 바꿔가며 3회까지 가본다.
ctx2 = FakeToolContext()
for q in ["신림역 조용한 작업 백반집", "신림역 백반집", "신림역 밥집"]:
    res = await collect_places(q, ctx2)
    print(res.get("status"), "|", q)

[collect_places] '신림역 조용한 작업 백반집' 카카오 0개 (시도 1/3, exhausted=True)
[collect_places] 총 0.1초
   kakao           0.1초  (1회, 평균 0.06초)
ok | 신림역 조용한 작업 백반집
[collect_places] '신림역 백반집' 카카오 45개 (시도 2/3, exhausted=True)
[collect_places] 총 0.2초
   kakao           0.2초  (3회, 평균 0.07초)
ok | 신림역 백반집
[collect_places] '신림역 밥집' 카카오 45개 (시도 3/3, exhausted=True)
[collect_places] 총 0.3초
   kakao           0.3초  (3회, 평균 0.10초)
ok | 신림역 밥집


In [49]:
# collect_info — 위 collect_places가 남긴 파일을 이어받는다.
# region_prefix는 도로명이 아니라 사람들이 실제로 쓰는 지역 통칭을 넣는다.
info = await collect_info(ctx, "신림역 혼밥하기 좋은 백반집", "신림")
print(info)

summaries = _load_json(info["summary_file"])
for name, s in list(summaries.items())[:3]:
    print(f"\n=== {name} ({s['review_count']}건) | {s['address']}")
    print("  강점:", s["strengths"])
    print("  약점:", s["weaknesses"])
    print("  요약:", s["summary"])

[collect_info] 후기 미확보 3개 → {'no_match_after_filter': 3}
[collect_info] 크롤링 181건 중 성공 181건
[collect_info] 총 28.4초
   scrape       1252.7초  (181회, 평균 6.92초)
   naver         289.4초  (80회, 평균 3.62초)
   llm            57.9초  (37회, 평균 1.56초)
{'count': 37, 'no_info': 3, 'url_file': 'tool_file\\1786620787035_info_urls.json', 'summary_file': 'tool_file\\1786620787035_info_summary.json'}

=== 신림춘천집 본점 (5건) | 서울 관악구 남부순환로 1600-8
  강점: ['신림역에서 도보 1분 거리로 접근성이 매우 뛰어남', '직원이 닭갈비를 직접 볶아주어 혼자서도 편하게 식사 가능', '셀프바를 통해 쌈채소와 밑반찬을 눈치 보지 않고 자유롭게 이용 가능', '태블릿 키오스크를 통한 비대면 주문 방식']
  약점: ['테이블 간격이 좁고 책상이 작아 혼자 식사 시 짐 보관이나 공간 활용이 다소 불편할 수 있음', '피크 타임에는 웨이팅이 발생할 수 있음', '주로 2인 이상 세트 메뉴 위주로 구성되어 있어 1인분 주문 가능 여부를 사전에 확인해야 함']
  요약: 신림역 인근에서 접근성이 좋고 직원이 직접 조리해 주는 편리함 덕분에 혼밥하기에 나쁘지 않은 환경입니다. 다만 매장이 붐비는 시간대에는 테이블이 좁고 대기가 있을 수 있어, 혼잡한 시간을 피해 방문하는 것을 권장합니다.

=== 원조민속순대타운 (5건) | 서울 관악구 신림로59길 14
  강점: ['1인분 주문이 가능하여 혼밥에 용이함', '가격 대비 양이 매우 푸짐함', '기본 서비스로 간과 음료수가 제공됨', '신림역에서 도보 3~5분 거리로 접근성이 좋음']
  약점: ['여러 가게가 모여 있는 개방형 구

In [50]:
# collect_context — 축제 케이스. 카카오로 위치를 잡고 웹·뉴스로 일정을 채운다.
# exhausted 처리 덕분에 결과가 적어도 retry 없이 한 번에 ok가 나온다.
ctx3 = FakeToolContext()
print(await collect_places("진해 군항제", ctx3))

context = await collect_context(ctx3, "진해 군항제 언제 열려?")
print(context)

contexts = _load_json(context["context_file"])
for name, c in list(contexts.items())[:3]:
    print(f"\n=== {name} ({c['source_count']}건, {c['freshness']})")
    print("  기간:", c["period"])
    print("  사실:", c["facts"])
    print("  요약:", c["summary"])

[collect_places] '진해 군항제' 카카오 6개 (시도 1/3, exhausted=True)
[collect_places] 총 0.1초
   kakao           0.1초  (1회, 평균 0.07초)
{'status': 'ok', 'count': 6, 'source_breakdown': {'local': 6, 'blog_derived': 0}, 'file': 'tool_file\\1786620816113_collect_places.json'}
[collect_context] 6곳 중 문서 확보 6곳
[collect_context] 크롤링 18건 중 성공 17건
[collect_context] 총 12.5초
   scrape         23.0초  (18회, 평균 1.28초)
   naver           8.7초  (12회, 평균 0.72초)
   llm             7.5초  (6회, 평균 1.26초)
{'count': 6, 'no_info': 0, 'source_file': 'tool_file\\1786620816113_context_sources.json', 'context_file': 'tool_file\\1786620816113_context_summary.json'}

=== 진해군항제 (4건, outdated)
  기간: 2026.03.27 ~ 2026.04.05
  사실: ['경상남도 창원시 진해구 일원에서 개최되는 국내 최대 벚꽃 축제임', '이충무공 추모제와 군악의장페스티벌, 벚꽃 명소 탐방 등 다양한 프로그램 운영', '평소 출입이 제한되는 해군사관학교와 해군진해기지사령부를 축제 기간에 개방함', '대부분 무료로 관람 가능하나 체리블라썸뮤직페스티벌 등 일부 프로그램은 유료로 운영됨']
  요약: 진해군항제는 충무공 이순신 장군의 얼을 기리고 벚꽃을 즐기는 대한민국 대표 봄 축제입니다. 2026년에는 3월 27일부터 4월 5일까지 개최되었으며, 군악의장페스티벌과 해군 시설 개방 등 군항도시만의 특색 있는 행사

In [51]:
# read_file — tool_file/ 밖은 막히는지 확인
import glob
latest = sorted(glob.glob("./tool_file/*_info_summary.json"))[-1]
print(read_file(latest)["content"].keys())
print(read_file("./wh_poc.ipynb"))    # 존재하지만 허용 경로 밖

dict_keys(['신림춘천집 본점', '원조민속순대타운', '막불감동', '오첨지', '서울밥집', '정담은보쌈 신림점', '강강술래 신림점', '서울갈비 신림점', '봄날의 햇살', '문성골족발집', '미나리생삼겹', '서울집 신림직영점', '신림목포수산', '대원갈비', '깡통닭갈비', '신림황소곱창', '뇌산마을 신림직영점', '황제갈비', '전라도시네마', '전주식당', '본화로', '서울뼛국', '강남곱창이야기 신림점', '돼봉삼겹살 신림점', '십원집 신림점', '부뚜막', '맛사랑쭈꾸미 신림점', '장인닭갈비 신림점', '신풍루곱창구이', '신림정', '시골장칼국수', '왕주먹', '독도해물탕 신림점', '흥부보쌈', '도하수산', '서림실비순대', '손칼국수잘하는집'])
{'status': 'error', 'message': '허용되지 않은 경로입니다.'}


## Layer 3 — 에이전트 통합

에이전트가 쿼리 성격에 따라 다른 툴을 고르는지가 관전 포인트다.
매장형은 `collect_info`, 행사형은 `collect_context`를 골라야 한다.

In [60]:
VERBOSE = True   # 에이전트 실행 중엔 툴 내부 로그를 끄고 툴 호출 흐름만 본다

out = await run("신림역에서 혼밥하기 좋은 한식집 추천해줘")

for p in (out["result"] or {}).get("places", []):
    print(f"\n■ {p['name']}  ({p['address']})")
    print(f"  {p['reason']}")
    print(f"  {p['highlights']}")
    if p["caution"]:
        print(f"  주의: {p['caution']}")

[CALL] get_today({})
[RESP] get_today → {'date': '2026-08-13', 'weekday': '목요일', 'time': '21:02', 'year': 2026, 'month': 8}
[CALL] collect_places({'query': '신림역 혼밥 한식', 'category_code': 'FD6'})
[collect_places] '신림역 혼밥 한식' 카카오 0개 (시도 1/3, exhausted=True)
[collect_places] 총 0.1초
   kakao           0.1초  (1회, 평균 0.07초)
[RESP] collect_places → {'status': 'ok', 'count': 0, 'source_breakdown': {'local': 0, 'blog_derived': 0}, 'file': 'tool_file\\1786622528778_collect_places.json'}
[CALL] collect_places({'category_code': 'FD6', 'query': '신림역 한식집'})
[collect_places] '신림역 한식집' 카카오 45개 (시도 2/3, exhausted=True)
[collect_places] 총 0.3초
   kakao           0.3초  (3회, 평균 0.10초)
[RESP] collect_places → {'status': 'ok', 'count': 40, 'source_breakdown': {'local': 40, 'blog_derived': 0}, 'file': 'tool_file\\1786622528778_collect_places.json'}
[CALL] collect_info({'user_query': '신림역 근처 혼밥하기 좋은 한식집 추천', 'region_prefix': '신림'})
[collect_info] 후기 미확보 2개 → {'no_match_after_filter': 2}
[collect_info] 크롤링 186건

In [53]:
# 행사형 쿼리 — collect_context를 고르는지 확인
out2 = await run("이번에 진해 군항제 가려는데 언제 하는지랑 볼거리 알려줘")
print([c["name"] for c in out2["tool_calls"]])

[CALL] get_today({})
[RESP] get_today → {'date': '2026-08-13', 'weekday': '목요일', 'time': '20:34', 'year': 2026, 'month': 8}
[CALL] collect_places({'query': '진해 군항제'})
[collect_places] '진해 군항제' 카카오 6개 (시도 1/3, exhausted=True)
[collect_places] 총 0.1초
   kakao           0.1초  (1회, 평균 0.06초)
[RESP] collect_places → {'status': 'ok', 'count': 6, 'source_breakdown': {'local': 6, 'blog_derived': 0}, 'file': 'tool_file\\1786620853795_collect_places.json'}
[CALL] collect_context({'user_query': '진해 군항제 일정 및 볼거리'})
[collect_context] 6곳 중 문서 확보 6곳
[collect_context] 크롤링 18건 중 성공 17건
[collect_context] 총 7.7초
   scrape         19.2초  (18회, 평균 1.07초)
   naver          11.0초  (12회, 평균 0.91초)
   llm             7.7초  (6회, 평균 1.29초)
[RESP] collect_context → {'count': 6, 'no_info': 0, 'source_file': 'tool_file\\1786620853795_context_sources.json', 'context_file': 'tool_file\\1786620853795_context_summary.json'}
[CALL] read_file({'file_path': 'tool_file\\1786620853795_context_summary.json'})
[RESP] read_fil

In [54]:
# 애매한 쿼리 — 둘 다 부르는지, 하나만 부르는지
out3 = await run("가평 캠핑장 추천해줘. 시설 좋고 예약 쉬운 곳으로")
print([c["name"] for c in out3["tool_calls"]])

[CALL] get_today({})
[RESP] get_today → {'date': '2026-08-13', 'weekday': '목요일', 'time': '20:34', 'year': 2026, 'month': 8}
[CALL] collect_places({'query': '가평 캠핑장'})
[collect_places] '가평 캠핑장' 카카오 45개 (시도 1/3, exhausted=True)
[collect_places] 총 0.2초
   kakao           0.2초  (3회, 평균 0.07초)
[RESP] collect_places → {'status': 'ok', 'count': 40, 'source_breakdown': {'local': 40, 'blog_derived': 0}, 'file': 'tool_file\\1786620867204_collect_places.json'}
[CALL] collect_info({'user_query': '시설 좋고 예약 쉬운 가평 캠핑장', 'region_prefix': '가평'})
[collect_info] 후기 미확보 3개 → {'no_match_after_filter': 3}
[collect_info] 크롤링 158건 중 성공 158건
[collect_info] 총 28.9초
   scrape       1016.9초  (158회, 평균 6.44초)
   naver         280.3초  (80회, 평균 3.50초)
   llm            60.6초  (37회, 평균 1.64초)
[RESP] collect_info → {'count': 37, 'no_info': 3, 'url_file': 'tool_file\\1786620867204_info_urls.json', 'summary_file': 'tool_file\\1786620867204_info_summary.json'}
[CALL] collect_context({'user_query': '시설 좋고 예약 쉬운 가평 캠핑장 추천 

In [57]:
out2 = await run("안녕 널 뭘하는 애임?")
print([c["name"] for c in out2["tool_calls"]])

[]


---
# 남은 과제

| 항목 | 현황 | 메모 |
|---|---|---|
| 후기 미확보율 | 개선 중 | `sim`+`date` 병합으로 줄었으나, '옛집'·'행복한식당' 같은 흔한 상호명은 여전히 취약 |
| `category_code` | 에이전트가 임의 지정 | 잘못 넣으면 결과가 급감. 닫을지 검토 |
| `MIN_COUNT` 20 고정 | 쿼리 유형 무관 | `exhausted`로 축제 케이스는 해결됐지만, 후보가 적당히 있는 애매한 경우는 여전히 retry |
| 리랭킹 | 에이전트가 직접 선별 | 후보 20~40개는 컨텍스트에 들어감. 100개 넘으면 별도 스코어링 필요 |
| 비네이버 도메인 | 셀렉터 폴백으로 일부 대응 | 공식 사이트는 봇 차단이 많아 스니펫 폴백에 의존 |
| 세마포어 전역 | 프로세스 단위 공유 | 서버화 시 요청 간 간섭 검토 |
| `collect_context` 실측 | 미검증 | 신규 코드. `freshness` 판정 정확도와 스니펫 폴백 비율 확인 필요 |